<a href="https://colab.research.google.com/gist/extramevoid-eng/3235c4707875387c0e04275c3ff4018d/mobilegs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Mobile-GS: End-to-End Training & WebGL Viewing in Colab
**Train and view 3D Gaussian Splats directly in your browser.**

This notebook provides a bulletproof, automated environment to train a Mobile-GS (Gaussian Splatting) model. It specifically patches several known C++ compilation errors and legacy Python bugs present in the original repository.

**Key Features:**
* 📦 **Isolated Environment:** Uses `uv` for ultra-fast dependency resolution.
* 🛠️ **Automated Patching:** Injects missing C++ headers (`<float.h>`, `glm`) and fixes legacy `numpy` datatypes (`np.byte` -> `np.uint8`).
* 🖥️ **Native WebGL Viewer:** Compresses the output to a `.splat` file and hosts it via Colab's native secure proxy (No Localtunnel/Cloudflare needed).

In [ ]:
#@title v-- Tap this if you are on Mobile { display-mode: "form" }
%%html
<b>Ensure this audio is playing to prevent colab from shutting down, then start tabs below</b><br/>
<audio autoplay="" src="https://raw.githubusercontent.com/KoboldAI/KoboldAI-Client/main/colab/silence.m4a" loop controls>

In [ ]:
# @title 1. Setup Environment & Fix Source Code
import os
import subprocess

print("🚀 Cloning Mobile-GS Repository...")
!git clone -q https://github.com/hbb1/Mobile-GS.git
%cd /content/Mobile-GS

print("⚡ Installing 'uv' (Ultra-fast Python package manager)...")
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1
os.environ['PATH'] = f"/root/.local/bin:/root/.cargo/bin:{os.environ['PATH']}"

print("📦 Creating isolated virtual environment...")
!uv venv /content/Mobile-GS/.venv

print("🛠️ Patching C++ Source Files...")
# Fix 1: Colab lacks specific math headers needed by the rasterizer
!apt-get install -y libglm-dev > /dev/null
!mkdir -p /content/Mobile-GS/submodules/diff-gaussian-rasterization/third_party/glm
!cp -r /usr/include/glm/* /content/Mobile-GS/submodules/diff-gaussian-rasterization/third_party/glm/

# Fix 2: simple_knn fails to compile because FLT_MAX is undefined in newer CUDA versions.
# We forcefully inject <float.h> into the top of the C++ file.
!sed -i '1i #include <float.h>' submodules/simple-knn/simple_knn.cu

print("📥 Installing Python dependencies (adding missing 'icecream')...")
!uv pip install --python /content/Mobile-GS/.venv -q -r requirements.txt
!uv pip install --python /content/Mobile-GS/.venv -q icecream matplotlib opencv-python tensorboard

print("⚙️ Compiling CUDA Kernels (This takes a minute)...")
!uv pip install --python /content/Mobile-GS/.venv -q -e submodules/diff-gaussian-rasterization
!uv pip install --python /content/Mobile-GS/.venv -q -e submodules/simple-knn

print("✅ Environment setup complete & kernels compiled!")

In [ ]:
# @title 1. Setup Environment & Fix Source Code
import os
import subprocess

print("🚀 Cloning Mobile-GS Repository...")
!git clone -q https://github.com/hbb1/Mobile-GS.git
%cd /content/Mobile-GS

print("⚡ Installing 'uv' (Ultra-fast Python package manager)...")
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1
os.environ['PATH'] = f"/root/.local/bin:/root/.cargo/bin:{os.environ['PATH']}"

print("📦 Creating isolated virtual environment...")
!uv venv /content/Mobile-GS/.venv

print("🛠️ Patching C++ Source Files...")
# Fix 1: Colab lacks specific math headers needed by the rasterizer
!apt-get install -y libglm-dev > /dev/null
!mkdir -p /content/Mobile-GS/submodules/diff-gaussian-rasterization/third_party/glm
!cp -r /usr/include/glm/* /content/Mobile-GS/submodules/diff-gaussian-rasterization/third_party/glm/

# Fix 2: simple_knn fails to compile because FLT_MAX is undefined in newer CUDA versions.
# We forcefully inject <float.h> into the top of the C++ file.
!sed -i '1i #include <float.h>' submodules/simple-knn/simple_knn.cu

print("📥 Installing Python dependencies (adding missing 'icecream')...")
!uv pip install --python /content/Mobile-GS/.venv -q -r requirements.txt
!uv pip install --python /content/Mobile-GS/.venv -q icecream matplotlib opencv-python tensorboard

print("⚙️ Compiling CUDA Kernels (This takes a minute)...")
!uv pip install --python /content/Mobile-GS/.venv -q -e submodules/diff-gaussian-rasterization
!uv pip install --python /content/Mobile-GS/.venv -q -e submodules/simple-knn

print("✅ Environment setup complete & kernels compiled!")

### 2. Download the Dataset
Downloading the standard NeRF synthetic "Lego" dataset.

In [ ]:
# @title 2. Download Lego Dataset
%cd /content/Mobile-GS
!mkdir -p data
%cd data
print("📥 Downloading NeRF Synthetic Dataset...")
!wget -qO nerf_synthetic.zip https://huggingface.co/datasets/YouLiXiya/nerf/resolve/main/nerf_synthetic.zip
print("📦 Extracting dataset...")
!unzip -q nerf_synthetic.zip
%cd /content/Mobile-GS
print("✅ Dataset ready at /data/nerf_synthetic/lego")

### 3. Patch Dataset Readers & Train
Before running the pretrain script, we must patch a legacy Pillow bug. Pillow no longer supports `np.byte`, which causes the dataset loader to crash. We update it to `np.uint8`. We also force `matplotlib` to run in headless mode to prevent UI crashes in Colab.

In [ ]:
# @title 3. Patch Python Bugs & Start Training
%cd /content/Mobile-GS

print("🩹 Patching legacy np.byte bugs in the dataset reader...")
!sed -i 's/dtype=np.byte/dtype=np.uint8/g' scene/dataset_readers.py

print("🔥 Starting 3D Gaussian Splatting Training (1,000 Iterations)...")
# Run training with headless Matplotlib (MPLBACKEND=Agg)
!export MPLBACKEND=Agg && ./.venv/bin/python pretrain.py \
  -s data/nerf_synthetic/lego \
  -m output/webgpu_lego_mini_pretrain \
  --eval \
  --imp_metric indoor \
  --sh_degree 3 \
  --iterations 1000 \
  -r 8 \
  --save_iterations 1000 \
  --checkpoint_iterations 1000 \
  --test_iterations 1000

print("🎉 Training Complete!")

### 4. Compress & Launch WebGL Viewer natively
This step bypasses the need for third-party tunnels (like Localtunnel or Cloudflare). It converts the heavy `.ply` file to a `.splat` file, hardcodes the WebGL viewer to read it, and hosts it securely via Google Colab's internal proxy.

In [ ]:
# @title 4. Compress & Launch Embedded Viewer
import os
import re
import subprocess
import time
from google.colab import output

print("📥 Cloning Antimatter15 WebGL Viewer...")
!rm -rf /content/splat
!git clone -q https://github.com/antimatter15/splat.git /content/splat

print("⚙️ Compressing .ply to optimized .splat format...")
!/content/Mobile-GS/.venv/bin/python /content/splat/convert.py \
  /content/Mobile-GS/output/webgpu_lego_mini_pretrain/point_cloud/iteration_1000/point_cloud.ply \
  /content/splat/lego.splat > /dev/null 2>&1

# Move the generated file to the web directory
!cp /content/Mobile-GS/output/webgpu_lego_mini_pretrain/point_cloud/iteration_1000/point_cloud.ply.splat /content/splat/lego.splat

print("🔧 Patching viewer to bypass HuggingFace defaults...")
with open('/content/splat/main.js', 'r') as f:
    content = f.read()
# Force the viewer to load our local file instead of the default demo
content = re.sub(r'"https://huggingface\.co[^"]+"', '"lego.splat"', content)
with open('/content/splat/main.js', 'w') as f:
    f.write(content)

print("🌐 Booting local server on port 8000...")
os.system("fuser -k 8000/tcp 2>/dev/null")
subprocess.Popen(["python3", "-m", "http.server", "8000", "-d", "/content/splat"])
time.sleep(2)

print("🔗 Generating Google Secure Link...")
try:
    proxy_url = output.eval_js("google.colab.kernel.proxyPort(8000)")
    print("\n" + "="*65)
    print("🚀 CLICK HERE TO VIEW ON MOBILE OR PC:")
    print(proxy_url)
    print("="*65 + "\n")

    print("🖼️ Loading iframe inside notebook...")
    output.serve_kernel_port_as_iframe(8000, height=600)
except Exception as e:
    print("Could not generate link. Ensure you are running this in a modern browser.")